# Assignment 2

In [42]:
import numpy as np

Least minimisation of $\|\underline{H_m}y-\|r_0\|e_1\|$ with a QR-decomposition of $\underline{H_m}$ :

In [45]:
def minimisation(Hm,beta):
    m = Hm.shape[1]
    e1 = np.zeros(m + 1)
    e1[0] = 1

    Q, R = np.linalg.qr(Hm)

    g = Q.T @ (beta*e1)

    y_m = np.linalg.solve(R, g[:m])

    return y_m

GMRES algorithm :

In [54]:
def GMRES(A, b, m, tol, max_iter):
    n = A.shape[0]
    x0 = np.zeros(n)
    r0 = b-A@x0
    beta = np.linalg.norm(r0)
    k = 0
    
    while beta >= tol and k < max_iter:
        V = np.zeros((n, m + 1))
        V[:, 0] = r0 / beta # v1 = r0/beta
        
        H = np.zeros((m+1,m))
        
        for j in range(m):
            w_j = A@V[:, j] # w_j = A*v_j
            for i in range(j):
                H[i,j] = np.dot(w_j,V[:, i]) # h_{i,j} = <w_j,v_i>
                w_j -= H[i,j]*V[:, i] # w_j = w_j - h_{i,j}v_i
            H[j+1,j] = np.linalg.norm(w_j) # h_{j+1,j} = ||w_j||
            if H[j + 1, j] != 0:
                V[:, j+1] = w_j/H[j+1,j] # v_{j+1} = w_j/h_{j+1,j}
        y_m = minimisation(H,beta) # y_m = argmin_y ||H_m*y - beta*e_1||
        x0 += V[:, :m]@y_m # Change initial value x0 = x = x0 + V_m*y_m
        r0 = b-A@x0
        beta = np.linalg.norm(r0)
        k+=1
    
    if k == max_iter:
        print("Maximum iterations")
    return x0, beta

Testing our algorithm on a "small" random matrix/vector

In [56]:
tol = 1e-6
max_iter = 1000

n = 5

A = np.random.rand(n,n)
b = np.random.rand(n)

m = 25

x,res = GMRES(A,b,m,tol,max_iter)

print("Residual norm =", res)

Residual norm = 9.608717013721678e-07


Testing our algorithm on a "large" random matrix/vector

In [60]:
tol = 1e-6
max_iter = 1000

n = 500

A = np.random.rand(n,n)
b = np.random.rand(n)

m = 2500

x, res = GMRES(A,b,m,tol,max_iter)

print("Residual norm =", res)

Residual norm = 2.265407282702989e-07
